In [3]:
%pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import time
import csv
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Configuración de entorno para Deepseek
def setup_environment_deepseek():
    """Carga las variables de entorno y retorna el API key para OpenRouter."""
    load_dotenv(override=True)
    openrouter_api_key = os.environ.get("OPENROUTER_API_KEY")
    if not openrouter_api_key:
        raise EnvironmentError("OPENROUTER_API_KEY no está definido en las variables de entorno.")
    # Para verificación parcial, se muestra el primer y últimos caracteres.
    print(openrouter_api_key[:1] + "..." + openrouter_api_key[-3:])
    return openrouter_api_key

# Clase para simular una sesión de chat utilizando Deepseek
class ChatSessionDeepseek:
    def __init__(self, client, model, extra_headers=None, extra_body=None):
        self.client = client
        self.model = model
        self.extra_headers = extra_headers or {
            "HTTP-Referer": "<YOUR_SITE_URL>",  # Opcional
            "X-Title": "<YOUR_SITE_NAME>"         # Opcional
        }
        self.extra_body = extra_body or {}
        
    def send_message(self, prompt_text: str):
        completion = self.client.chat.completions.create(
            extra_headers=self.extra_headers,
            extra_body=self.extra_body,
            model=self.model,
            messages=[{"role": "user", "content": prompt_text}],
        )
        # Se crea un objeto respuesta simple para mantener compatibilidad con el resto del código
        class Response:
            pass
        response_obj = Response()
        response_obj.text = completion.choices[0].message.content
        # Si se requiriesen métricas, se podría extender este objeto; en este ejemplo se deja vacío.
        response_obj.candidates = []
        return response_obj

def create_chat_session_deepseek(model: str) -> ChatSessionDeepseek:
    """Crea y retorna una sesión de chat utilizando Deepseek."""
    openrouter_api_key = setup_environment_deepseek()
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=openrouter_api_key,
    )
    return ChatSessionDeepseek(client, model)

def read_problems_from_csv(input_filename: str):
    """Lee el archivo CSV y retorna una lista de problemas."""
    problems = []
    with open(input_filename, mode='r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)
        next(csv_reader, None)  # Salta la cabecera si existe
        for fields in csv_reader:
            if len(fields) < 2:
                continue
            problem = {"ID": fields[0], "Description": fields[1]}
            problems.append(problem)
    return problems

def extract_code(response_text: str) -> str:
    """
    Extrae el código Python del texto de respuesta, eliminando bloques markdown.
    Si se encuentra un bloque '```python' o '```', se extrae el contenido interno.
    """
    if "```python" in response_text:
        parts = response_text.split("```python")
        if len(parts) > 1:
            code_content = parts[1].split("```")[0].strip()
            return code_content
    elif "```" in response_text:
        parts = response_text.split("```")
        if len(parts) > 1:
            return parts[1].strip()
    return response_text.strip()

def process_problem(problem: dict, index: int, chat_session, model: str, output_directory: str) -> dict:
    """
    Procesa un problema:
      - Construye el prompt y envía la solicitud al modelo Deepseek.
      - Extrae y limpia el código Python de la respuesta.
      - Guarda el código en un archivo .py.
      - Retorna un diccionario con las métricas y el resultado.
    """
    prompt_text = (
        f"Write a Python solution for the following problem:\n\n"
        f"{problem['Description']}\n\n"
        "Provide only executable Python code, no explanations."
    )
    response = chat_session.send_message(prompt_text)
    python_code = extract_code(response.text)
    
    # Guarda el código generado en un archivo .py
    output_file = Path(output_directory) / f"output_{index + 1}.py"
    with open(output_file, mode='w', encoding='utf-8') as f:
        f.write(python_code)
    print(f"Python code saved successfully in {output_file}.")
    
    # Se generan algunas métricas básicas
    code_metrics = {
        "problem_id": index + 1,
        "code_length": len(python_code),
        "model": model,
        "input_tokens": len(prompt_text.split())
    }
    
    return {
        "ID": index + 1,
        "code": python_code,
        "result": "SUCCESS",
        "true_count": code_metrics["code_length"],
        "false_count": 0
    }

def main():
    try:
        # Configuración y creación de la sesión de chat con Deepseek
        deepseek_model = "deepseek/deepseek-r1:free"
        chat_session = create_chat_session_deepseek(deepseek_model)
        deepseek_model_name = deepseek_model.replace(":", "_").replace("/", "_")[:-5]
        # Configuración de rutas y parámetros
        temperature_value = 1  # Valor usado para organizar directorios (puede ignorarse en Deepseek)
        input_filename = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\data\processed\leetcode_problems_processed_data.csv"
        output_directory = rf"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\py_files_outputs_v2\temperature-{temperature_value}\{deepseek_model_name}"
        results_directory = rf"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\results\temperature-{temperature_value}\{deepseek_model_name}"
        
        # Crear directorios si no existen
        Path(output_directory).mkdir(parents=True, exist_ok=True)
        Path(results_directory).mkdir(parents=True, exist_ok=True)
        
        # Lectura de los problemas
        problems = read_problems_from_csv(input_filename)
        if not problems:
            print("No se encontraron problemas en el archivo CSV.")
            return
        
        start_index = 0
        max_problems = 15
        results = []
        
        # Procesa cada problema
        for i in range(start_index, min(len(problems), start_index + max_problems)):
            print(f"Processing Problem {i + 1} of {max_problems}")
            start_time = time.time()
            try:
                result = process_problem(problems[i], i, chat_session, deepseek_model, output_directory)
                results.append(result)
            except Exception as e:
                print(f"Error processing problem {i + 1}: {e}")
                results.append({
                    "ID": i + 1,
                    "code": "",
                    "result": str(e),
                    "true_count": "ERROR",
                    "false_count": "ERROR"
                })
            elapsed_time = time.time() - start_time
            # Pausa para respetar límites de tasa (ajustado a 60 segundos en este ejemplo)
            if i < min(len(problems), start_index + max_problems) - 1:
                time.sleep(max(0, 60 - elapsed_time))
        
        # Guardar resultados en un archivo CSV
        results_csv = Path(results_directory) / f"results_{deepseek_model.replace('/', '_')}.csv"
        with open(results_csv, mode='w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=["ID", "code", "result", "true_count", "false_count"])
            writer.writeheader()
            writer.writerows(results)
        print(f"CSV file '{results_csv}' created successfully.")
    
    except Exception as ex:
        print(f"An error occurred: {ex}")

if __name__ == "__main__":
    main()


s...779
Processing Problem 1 of 15
Python code saved successfully in C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\py_files_outputs_v2\temperature-1\deepseek_deepseek-r1\output_1.py.
Processing Problem 2 of 15
Python code saved successfully in C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\py_files_outputs_v2\temperature-1\deepseek_deepseek-r1\output_2.py.
Processing Problem 3 of 15
Python code saved successfully in C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\py_files_outputs_v2\temperature-1\deepseek_deepseek-r1\output_3.py.
Processing Problem 4 of 15
Python code saved successfully in C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\py_files_outputs_v2\temperature-1\deepseek_deepseek-r1\output_4.py.
Processing Problem 5 of 15


: 